In [0]:
%pip install -U --quiet databricks-sdk==0.49.0 "databricks-langchain==0.4.0" databricks-agents mlflow[databricks] databricks-vectorsearch langchain==0.3.25 markdownify pydantic==2.10.1 mlflow openai PyMuPDF

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,
    max_tokens=250,
)

chat_model.invoke("Who is data fiduciary?")

In [0]:
import fitz # PyMuPDF
file_path='/Volumes/workspace/default/rag_data/demoVol/dpact.pdf'
# Read file content
with open(file_path, "rb") as f:
    pdf_bytes = f.read()
# Use PyMuPDF to extract text
doc = fitz.open("pdf", pdf_bytes)
text = ""
for page in doc:
    text += page.get_text()
print(text) # Print first 1000 characters

In [0]:
%sql
drop table workspace.default.pdf_table

In [0]:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()
df = spark.createDataFrame([(text,)], ["text"])
df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.pdf_table")

In [0]:
df = spark.table("workspace.default.pdf_table")

df_clean = df.filter(df.text.isNotNull() & (df.text != "") & (~df.text.contains("End of Page")))

In [0]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os

# Load documents (e.g., text files or PDFs from DBFS or local)
raw_text = text
# Replace this with your actual file reader logic

# Chunk documents for embedding
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.create_documents([raw_text])

import pandas as pd
# Convert docs to a list of dicts for display
pd_docs = pd.DataFrame([doc.dict() for doc in docs])

pd_docs.insert(0, "id_pk", range(1, len(pd_docs) + 1))

display(pd_docs)

In [0]:
len(docs)

In [0]:
docs[0]

In [0]:
import pandas as pd
# Convert docs to a list of dicts for display
pd_docs = pd.DataFrame([doc.dict() for doc in docs])

pd_docs.insert(0, "id_pk", range(1, len(pd_docs) + 1))

display(pd_docs)

In [0]:
spark_df = spark.createDataFrame(pd_docs[['id_pk', 'page_content']])

spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.my_pages2")

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

In [0]:
client.list_endpoints()


In [0]:
# Set table and catalog
index_name = "workspace.default.documents_index"

# Get or create the index
try:
    vs_index = client.get_index(index_name=index_name)
    print(f"Using existing index: {index_name}")
except Exception:
    vs_index = client.create_delta_sync_index(
        endpoint_name="vector_search",
        index_name=index_name,
        primary_key="id_pk",
        source_table_name="workspace.default.my_pages2",
        pipeline_type="TRIGGERED",
        embedding_source_column="page_content",
        embedding_model_endpoint_name="databricks-gte-large-en"
    )
    print(f"Created new index: {index_name}")

In [0]:
client.list_indexes('vector_search')

In [0]:
index = client.get_index(index_name="workspace.default.documents_index")

In [0]:
# Get or create the index if it doesn't exist
index_name = "workspace.default.documents_index"

try:
    index = client.get_index(index_name=index_name)
except Exception:
    # Index doesn't exist, create it
    index = client.create_delta_sync_index(
        endpoint_name="vector_search",
        index_name=index_name,
        primary_key="id_pk",
        source_table_name="workspace.default.my_pages2",
        pipeline_type="TRIGGERED",
        embedding_source_column="page_content",
        embedding_model_endpoint_name="databricks-gte-large-en"
    )
    print(f"Created index: {index_name}")

# Wait for index to be ready
index.wait_until_ready()

results_dict = index.similarity_search(
    query_text="Data Fudiciary?",
    columns=["id_pk", "page_content"],
    num_results=3
)

display(results_dict)

In [0]:
input = "Answer Question " + str(results_dict) + " Question: Who is the data fiduciary?"
print(input)

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,
    max_tokens=250,
)

chat_model.invoke(str(input))